# Capstone — mirrors your deployed research paper

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Question

*The research question and the decision it supports.*

**Question:** Can content items in the warehouse be grouped into a few performance archetypes, using only CTR, average position and impressions, that still hold up on clients the model never saw — and can each archetype be mapped to a sensible review action?

**Decision it supports:** Where should an editorial / SEO team spend limited review hours next quarter: fix titles, look closer at a page, or leave it alone?

**Who acts:** Content operations leads (assign the work) and SEO strategists (do the title / meta / linking work).

#### Cost of a wrong call.

- Prune a page that was really earning traffic → traffic lost overnight.
- Rewrite a whole article when a title tweak was enough → expensive hours wasted.

In [1]:
import os, json, warnings
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, f_oneway
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, silhouette_samples
warnings.filterwarnings('ignore')

SEED = 42
SIL_N = 20000
np.random.seed(SEED)
os.makedirs('work/outputs/figures', exist_ok=True)

def find(name):
    for d in ['work/outputs', '/content/drive/MyDrive', '.']:
        p = os.path.join(d, name)
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f"{name} not found - run w07 first or upload it next to this notebook.")

ACTION_MAP = {
    'High-Reach Opportunity':  ('FIX_TITLE_META_PRIORITIZED',  'RC_HIGH_REACH_CONFIRMED_GAP'),
    'Low-Visibility / Buried': ('REVIEW_FOR_PRUNE_OR_REWORK',  'RC_WORST_POSITION_LOW_SIGNAL'),
    'Steady / Low Signal':     ('LOW_PRIORITY_REVIEW',         'RC_MODEST_ALL_AXES'),
    'Already Converting':      ('MONITOR_ONLY',                'RC_HIGH_CTR_NO_GAP'),
}
for a, (act, rc) in ACTION_MAP.items():
    print(f"{a:26s} -> {act:28s} ({rc})")

High-Reach Opportunity     -> FIX_TITLE_META_PRIORITIZED   (RC_HIGH_REACH_CONFIRMED_GAP)
Low-Visibility / Buried    -> REVIEW_FOR_PRUNE_OR_REWORK   (RC_WORST_POSITION_LOW_SIGNAL)
Steady / Low Signal        -> LOW_PRIORITY_REVIEW          (RC_MODEST_ALL_AXES)
Already Converting         -> MONITOR_ONLY                 (RC_HIGH_CTR_NO_GAP)


## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

**Release:** FlyRank internship warehouse, build v20260703 (pseudonymized, gated).

**Table used:** fact_content_daily_performance only — Search Console columns: impressions, clicks, average position.

**Window:** Report dates 2025-01-27 to 2026-06-30. Each item is summed over its own available history (history depth differs by client) — not a Jan–Jun 2026 window.

One row = one content item (content_hash_id), one aggregate over its history.

**Kept:** Daily rows with impressions > 0, then items with at least 100 impressions in total → 182,135 items from 59 clients (the release lists 104 clients; the rest have too little search history to pass the filter).

**Excluded:**
- GA4 columns — zero-filled before each client's GA4 start; not needed for CTR / position / volume.
- fact_content_query_90d — its 90-day window overlaps the most recent months; skipping it avoids window-overlap leakage.
- dim_content / dim_clients — no field from them is a feature.
- Items under 100 impressions — CTR ratios are too noisy to act on (w04).

**Unit note:** In the warehouse aggregate, CTR is a fraction (0.0023 = 0.23%) — do not mix it with the ×100-percentage starter CSV.

**IDs (content_hash_id, client_hash_id) are pseudonyms:** grouping and splitting only, never features.

In [3]:
from google.colab import drive
drive.mount('/content/drive')

df = pd.read_csv(find('w07_action_playbook_queue.csv'))
summary = json.load(open(find('w07_playbook_summary.json')))

print(f"Rows: {len(df):,}   Clients: {df['client_hash_id'].nunique()}")
print(f"Impressions per item: min {df['impressions'].min():.0f}, median {df['impressions'].median():.0f}, max {df['impressions'].max():,.0f}")

dups = df['content_hash_id'].duplicated().sum()
print(f"Duplicate content_hash_id rows: {dups}")
print(f"Missing values in ctr / avg_position / impressions: "
      f"{int(df[['ctr','avg_position','impressions']].isna().sum().sum())}")
print(f"Items with avg_position == 0: {(df['avg_position'] == 0).sum()}")

assert len(df) == summary['n_items'] == 182135
assert df['client_hash_id'].nunique() == 59
assert dups == 0 and df['impressions'].min() >= 100

Mounted at /content/drive
Rows: 182,135   Clients: 59
Impressions per item: min 100, median 1513, max 2,902,616
Duplicate content_hash_id rows: 0
Missing values in ctr / avg_position / impressions: 0
Items with avg_position == 0: 1


In [ ]:
try:
    from google.colab import userdata
    import duckdb
    con = duckdb.connect()
    con.execute("CREATE SECRET (TYPE huggingface, TOKEN ?)", [userdata.get('HF_TOKEN')])
    base = 'hf://datasets/FlyRank/internship-warehouse'
    print(con.sql(f"""SELECT COUNT(*) AS rows, MIN(report_date) AS first_day, MAX(report_date) AS last_day
                      FROM read_parquet('{base}/fact_content_daily_performance/**/*.parquet')""").df())
    print(con.sql(f"SELECT COUNT(*) AS clients FROM read_parquet('{base}/dim_clients.parquet')").df())
except Exception as e:
    print("Warehouse check skipped (no Colab secret / no access):", type(e).__name__)

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

## 5. Limitations

*What this work cannot claim.*

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.